<a href="https://colab.research.google.com/github/Anasuki0310/DeepLearning/blob/main/Model_Deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1กำหนดรูปแบบการแปลงภาพ (Data Transformations)**ทำข้อความเป็นตัวหนา**

In [ ]:
import cv2
import os
from pathlib import Path

source_dir = Path("/content/dataset/original")
target_dir = Path("/content/dataset/shape_only")
count = 0

for img_path in source_dir.rglob("*.*"):
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue

        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # ปรับค่า Blur และ Canny ตามที่คุณหาจุดสมดุลได้
        img_blur = cv2.GaussianBlur(img_gray, (7, 7), 0)
        edges = cv2.Canny(img_blur, 50, 150)

        relative_path = img_path.relative_to(source_dir)
        save_path = target_dir / relative_path

        save_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(save_path), edges)
        count += 1

print(f"เสร็จสิ้น! แปลงภาพทั้งหมด {count} รูป ไว้ที่: {target_dir}")

เสร็จสิ้น! แปลงภาพทั้งหมด 275 รูป ไว้ที่: /content/dataset/shape_only


## ล็อกความสุ่ม (Reproducibility) & **นำเข้าไลบรารี**

In [ ]:
import torch
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # ล็อคการคำนวณของการ์ดจอให้ผลลัพธ์คงที่
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# เรียกใช้คำสั่งล็อคผลลัพธ์
set_seed(42)

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"กำลังใช้งาน: {device}") # ควรจะขึ้นว่า cuda

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# โหลดข้อมูล
data_dir = "/content/dataset/shape_only"
dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

class_names = dataset.classes
num_classes = len(class_names)
print(f"พบจำนวนคลาส: {num_classes} คลาส ({class_names}) | จำนวนรูปภาพรวม: {len(dataset)} รูป")

# โหลดโมเดลและปรับ Layer สุดท้าย
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

print("เตรียมโมเดล ResNet18 พร้อมเทรนบน GPU แล้ว!")

กำลังใช้งาน: cuda
พบจำนวนคลาส: 2 คลาส (['class_1', 'class_2']) | จำนวนรูปภาพรวม: 275 รูป
เตรียมโมเดล ResNet18 พร้อมเทรนบน GPU แล้ว!


In [ ]:
import torch.optim as optim
import time

# 1. กำหนด Loss Function และ Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # ใช้ Adam เพื่อให้เรียนรู้ไวขึ้น

num_epochs = 5 # จำนวนรอบในการรันข้อมูลซ้ำ (ลอง 5 รอบก่อน)

print("เริ่มกระบวนการฝึกสอน (Training)...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    model.train() # ตั้งค่าโมเดลให้อยู่ในโหมด Training

    running_loss = 0.0
    running_corrects = 0

    # วนลูปอ่านข้อมูลทีละ Batch (ทีละ 16 รูปตามที่เราตั้งไว้)
    for inputs, labels in dataloader:
        # ย้ายข้อมูลรูปภาพและคำตอบไปไว้บน GPU
        inputs = inputs.to(device)
        labels = labels.to(device)

        # เคลียร์ค่า Gradient สะสมจากรอบก่อนหน้า
        optimizer.zero_grad()

        # Forward pass: ให้โมเดลทายผล
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        # คำนวณค่าความผิดพลาด (Loss)
        loss = criterion(outputs, labels)

        # Backward pass: ปรับปรุงค่าน้ำหนัก (Backpropagation)
        loss.backward()
        optimizer.step()

        # เก็บสถิติผลการเทรน
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    # สรุปผลความแม่นยำประจำ Epoch
    epoch_loss = running_loss / len(dataset)
    epoch_acc = running_corrects.double() / len(dataset)

    print(f'Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}\n')

time_elapsed = time.time() - start_time
print(f'เทรนเสร็จสิ้นในเวลา {time_elapsed // 60:.0f} นาที {time_elapsed % 60:.0f} วินาที')

เริ่มกระบวนการฝึกสอน (Training)...
Epoch 1/5
----------
Loss: 0.9002 | Accuracy: 0.6655

Epoch 2/5
----------
Loss: 0.5126 | Accuracy: 0.7527

Epoch 3/5
----------
Loss: 0.2840 | Accuracy: 0.8909

Epoch 4/5
----------
Loss: 0.2198 | Accuracy: 0.9127

Epoch 5/5
----------
Loss: 0.3474 | Accuracy: 0.8545

เทรนเสร็จสิ้นในเวลา 0 นาที 10 วินาที


In [ ]:
import torch.nn.functional as F
from PIL import Image

# ฟังก์ชันสำหรับให้โมเดลทำนายภาพ 1 ภาพ
def test_cue_conflict(image_path, model_to_test, model_name):
    image = Image.open(image_path).convert('RGB')
    input_tensor = data_transforms(image).unsqueeze(0).to(device)

    model_to_test.eval() # ปิดโหมดเทรน
    with torch.no_grad():
        outputs = model_to_test(input_tensor)
        probabilities = F.softmax(outputs, dim=1)
        top_prob, top_class = torch.max(probabilities, 1)

    print(f"[{model_name}] ทำนายว่าเป็น: {class_names[top_class[0]]} (ความมั่นใจ {top_prob[0].item()*100:.2f}%)")

# --- วิธีใช้งาน ---
test_image = "/content/cat_zebra.jpg" # เปลี่ยนชื่อไฟล์เป็นรูปภาพขัดแย้งของคุณ

print("=== ผลการทดสอบภาพ Cue Conflict ===")

# โหลดและทดสอบด้วย Baseline Model (โมเดลปกติ)
model.load_state_dict(torch.load('/content/resnet18_baseline.pth'))
test_cue_conflict(test_image, model, "Baseline Model")

# โหลดและทดสอบด้วย Shape-biased Model (โมเดลลายเส้น)
model.load_state_dict(torch.load('/content/resnet18_shape_biased.pth'))
test_cue_conflict(test_image, model, "Shape-Biased Model")

=== ผลการทดสอบภาพ Cue Conflict ===
[Baseline Model] ทำนายว่าเป็น: class_1 (ความมั่นใจ 71.78%)
[Shape-Biased Model] ทำนายว่าเป็น: class_2 (ความมั่นใจ 81.37%)
